# పాఠం 09 - మెటాకాగ్నిషన్ డిజైన్ నమూనా


## సెటప్

ఈ నోట్బుక్ Microsoft Agent Framework ఉపయోగించి Metacognition డిజైన్ ప్యాటర్న్‌ను ప్రదర్శిస్తుంది.

**ముందస్తు అవసరాలు:**
- Azure OpenAI deployment ను environment variables ద్వారా కాన్ఫిగర్ చేయాలి
- Azure CLI ధృవీకరించబడినది (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## మెటాకాగ్నిషన్ అంటే ఏమిటి?

మెటాకాగ్నిషన్ అనగా **ఆలోచన గురించి ఆలోచించడం**. AI ఏజెంట్ల పరంగా, ఇది ఇలా చేయగల ఏజెంట్లను నిర్మించడం అని అర్థం:

- **స్వయంగా ప్రతిబింబించడం** తమ స్వంత అవుట్‌పుట్‌లు మరియు తార్కిక ప్రక్రియపై
- **లోపాలను గుర్తించడం** మరియు మౌనంగా విఫలమవకుండా క్రమంగా పునరుద్ధరించడం
- **అంచనా వేయడం** తమ సమాధానాలు పూర్తి, ఉపయోగకరమైనవేనా కాదా
- **అనుకూలీకరించుకోవడం** మొదటి పద్ధతి పని చేయకపోతే తమ వ్యూహాన్ని మార్చుకోవడం (ఉదాహరణకు, బ్యాకప్ సిస్టమ్‌కి తిరిగి వెళ్లటం)

మెటాకాగ్నిటివ్ ఏజెంట్ కేవలం ప్రశ్నలకు సమాధానం ఇవ్వదు — అది తన పనితీరును పర్యవేక్షించి తక్షణమే సర్దుబాటు చేసుకుంటుంది.


## ప్రాథమిక మరియు బ్యాకప్ సాధనాలు

ఒక సాధారణ మెటాకాగ్నిషన్ నమూనా **ఫాల్‌బ్యాక్ వ్యూహం**. ఏజెంట్ ముందుగా ఒక ప్రాథమిక సాధనాన్ని ప్రయత్నిస్తుంది; అది విఫలమైతే (ఉదా., 404 లోపం), ఏజెంట్ విఫలతను గుర్తించి పారదర్శకంగా ఒక బ్యాకప్ సాధనానికి మారిపోతుంది.

ఇది నిజజీవిత వ్యవస్థలను ప్రతిబింబిస్తుంది, అక్కడ ప్రాథమిక సేవలు అందుబాటులో లేకపోవచ్చు మరియు ఏజెంట్లు ప్రత్యామ్నాయ మార్గం ఎంచుకోవడానికి ముందుగా సమస్యను స్వయంగా నిర్ధారించుకోవాలి.

క్రింద మేము రెండు విమాన శోధన సాధనాలను నిర్వచిస్తున్నాం:
- **ప్రాథమिक** — పారిస్, టోక్యో, మరియు బార్సిలోనా ని కవర్ చేస్తుంది
- **బ్యాకప్** — బెర్లిన్, సిడ్నీ, మరియు న్యూయార్క్ సిటీ ని కవర్ చేస్తుంది


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## లోప పునరుద్ధరణతో స్వీయ-పరిశీలన ఏజెంట్

కిందివున్న ఏజెంట్‌కు మొదటగా ప్రాథమిక ఫ్లైట్ సిస్టమ్‌ను ప్రయత్నించాలని, విఫలతలను గుర్తించి పారదర్శకంగా బ్యాకప్ సిస్టమ్‌కి తిరిగి వెళ్లాలని సూచించబడింది. ప్రతి స్పందన తర్వాత అది సంక్షిప్తంగా స్వయంగా ఆత్మమూల్యాంకనం చేసి, వినియోగదారుడి ప్రశ్నకు అది పూర్తిగా సమాధానం ఇచ్చిందా లేదా తెలుసుకుంటుంది.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## స్వీయ-మూల్యాంకనం నమూనా

మెటాకాగ్నిషన్ యొక్క మరో కోణం **స్వీయ-మూల్యాంకనం**: ఒక వేరే ఏజెంట్ (లేదా రెండవ సారి అదే ఏజెంట్) ప్రతిస్పందనను సంపూర్ణత, ఖచ్చితత్వం, మరియు సహాయకత కోసం సమీక్షిస్తుంది.

కింది భాగంలో మనం మూడు పరిమాణాలపై ప్రయాణ ఏజెంట్ ప్రతిస్పందనలకు స్కోరు ఇచ్చే ఒక `ResponseEvaluator` ఏజెంట్‌ను సృష్టిస్తాము.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## సారాంశం

ఈ పాఠంలో మీరు Microsoft Agent Framework ను ఉపయోగించి **మెటాకాగ్నిటివ్ ఏజెంట్లు** ఎలా నిర్మించాలో నేర్చుకున్నారు:

- **స్వీయ-పరిశీలన**: తమ స్వంత నిర్ణయ ప్రక్రియను పర్యవేక్షించి ఏమి జరిగిందో పారదర్శకంగా తెలియజేసే ఏజెంట్లు.
- **ఫాల్‌బ్యాక్‌లతో లోపాల పునరుద్ధరణ**: ఏజెంట్ వైఫల్యాలను (ఉదాహరణకు, 404 లోపాలు) గుర్తించి స్వయంచాలకంగా ప్రత్యామ్నాయ మూలాన్ని ప్రయత్నించే ప్రాథమిక + బ్యాకప్ టూల్ ప్యాటర్న్.
- **స్వీయ-మూల్యాంకనం**: సంపూర్ణత, ఖచ్చితత్వం మరియు సహాయకత కోసం ప్రతిస్పందనలను స్కోర్ చేసే వేరే మూల్యాంకన ఏజెంట్.

ఈ నమూనాలు ఏజెంట్లను మరింత బలమైన, పారదర్శకమైన మరియు విశ్వసనీయమైనలా చేస్తాయి — ఉత్పత్తి అమలులకు ముఖ్యమైన లక్షణాలు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
బాధ్యతా నిరాకరణ:
ఈ పత్రాన్ని AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ద్వారా అనువదించబడింది. మేము ఖచ్చితత్వానికి యత్నిస్తున్నప్పటికీ, ఆటోమేటిక్ అనువాదాలలో తప్పులు లేదా అసంపూర్ణతలు ఉండవచ్చని దయచేసి గమనించండి. అసలైన పత్రాన్ని దాని స్థానిక భాష‌లో ఉన్నది అధికారిక మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం వృత్తిపరమైన మానవ అనువాదం సిఫార్సు చేయబడుతుంది. ఈ అనువాదాన్ని ఉపయోగించడం వల్ల ఏర్పడే ఏవైనా అపార్థాలు లేదా తప్పుగా అర్థం చేసుకోవడాలకై మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
